## Structural identifiability algorithm.

### Birth-death model as an example

The transition rates of the birth-death process are given by,

$$
\begin{align*}
 T_{\mathbf{n}}^{\mathbf{n} + \mathbf{z}_1} & = \theta_1 n_1 \\
T_{\mathbf{n}}^{\mathbf{n} + \mathbf{z}_2} & = \theta_2 n_1 \\
\end{align*}
$$
with stoichiometries $\mathbf{z}_1 = [+1,0]$ and $\mathbf{z}_2 = [-1,+1]$.
Here $n_1$ and $n_2$ are live and dead cells, and $\theta_1$ and $\theta_2$ the birth and death rates.

**Our objective is to code arbitrary ODE models (starting from transition rates) as a presence/absence matrix of parameters/variables in the ODEs and use it to assess the Structural Identifiability of the parameters.**

For the birth-death model above, we can code the transition rates and stoichiometries in the table:
&nbsp; | $θ₁$ | $\theta_2$ | $n_1$ | $n_2$ | $\mathbf{z}$
-|-|-|-|-|-
$T_{\mathbf{n}}^{\mathbf{n} + \mathbf{z}_1}$ | 1 | - | 1 | - | +1, 0
$T_{\mathbf{n}}^{\mathbf{n} + \mathbf{z}_2}$ | - | 1 | 1 | - | -1, +1 

where 1 indicates presence and - indicates absence of the parameter/variable.

In our code below we use `Catalyst.jl` to let users input transition rates using standard chemical kinetics notation.

#### Step 1: Dynamic equations derivation

We will write equations for the statistical moments (here 1st and 2nd) of each variable.

The notation for moments is $m_{\circ \circ}$, where each digit of the subindex corresponds to a variable in the system, and its value to the order of that variable in the moment.
For example, the moment $\langle n_1^3 n_2^2 \rangle$ is coded as $m_{32}$.

We will use `MomentClosure.jl` to automatically derive the moments (raw or central) ODEs from the transitions rates specified in `Catalyst.jl` and close them using community-accepted closures.
This is the set of ODEs for the birth-death model above, 
$$
\begin{align*}
\frac{dm_{1 0}}{dt} &= \lim_{\Delta t \to 0} \frac{m_{1 0}^{(t+\Delta t)} - m_{1 0}^{(t)}}{\Delta t} = m_{1 0}^{(t)} (\theta_1 - \theta_2) \\
\frac{dm_{0 1}}{dt} &= \lim_{\Delta t \to 0} \frac{m_{0 1}^{(t+\Delta t)} - m_{0 1}^{(t)}}{\Delta t} = m_{1 0}^{(t)} \theta_2 \\
\frac{dm_{2 0}}{dt} &= \lim_{\Delta t \to 0} \frac{m_{2 0}^{(t+\Delta t)} - m_{2 0}^{(t)}}{\Delta t} = 2 m_{2 0}^{(t)}(\theta_1 - \theta_2) + m_{1 0}^{(t)} (\theta_1 + \theta_2) \\
\frac{dm_{0 2}}{dt} &= \lim_{\Delta t \to 0} \frac{m_{0 2}^{(t+\Delta t)} - m_{0 2}^{(t)}}{\Delta t} = 2 m_{1 1}^{(t)} \theta_2 + m_{1 0}^{(t)} \theta_2 \\
\frac{dm_{1 1}}{dt} &= \lim_{\Delta t \to 0} \frac{m_{1 1}^{(t+\Delta t)} - m_{1 1}^{(t)}}{\Delta t} = m_{1 1}^{(t)} (\theta_1 - \theta_2) + m_{2 0}^{(t)} \theta_2 - m_{1 0}^{(t)} \theta_2

\end{align*}
$$

Without `MomentClosure.jl`, doing this by hand requires using the stoichiometries ($\mathbf{z}$) to determine which transitions are involved in each moment ODE (e.g. moments with only variable 1 ($\dot{m}_{10}$ and $\dot{m}_{20}$) involve $T_{\mathbf{n}}^{\mathbf{n} + \mathbf{z}_1}$ and $T_{\mathbf{n}}^{\mathbf{n} + \mathbf{z}_2}$, moments with only variable 2 ($\dot{m}_{01}$ and $\dot{m}_{02}$) involve $T_{\mathbf{n}}^{\mathbf{n} + \mathbf{z}_2}$ only, and moments with both variables ($\dot{m}_{11}$) involve $T_{\mathbf{n}}^{\mathbf{n} + \mathbf{z}_1}$ and $T_{\mathbf{n}}^{\mathbf{n} + \mathbf{z}_2}$). 

The matrix below is our resulting presence/absence matrix to assess Structural Identifiability.
You can compare this with the ODEs above.

&nbsp; | $\theta_1$ | $\theta_2$ | $m_{10}^{t_i}$ | $m_{01}^{t_i}$ | $m_{20}^{t_i}$ | $m_{02}^{t_i}$ | $m_{11}^{t_i}$ | $m_{10}^{t_{i+1}}$ | $m_{01}^{t_{i+1}}$ | $m_{20}^{t_{i+1}}$ | $m_{02}^{t_{i+1}}$ | $m_{11}^{t_{i+1}}$
-|-|-|-|-|-|-|-|-|-|-|-|-
$\dot{m}_{10}$ | 1 | -1 | 1 | - | - | - | - | 1 | - | - | - | -
$\dot{m}_{01}$ | - | 1 | 1 | 1 | - | - | - | - | 1 | - | - | -
$\dot{m}_{20}$ | (2, 1) | (-2, 1) | 1 | - | 1 | - | - | - | - | 1 | - | -
$\dot{m}_{02}$ | - | (2, 1) | 1 | - | - | 1 | 1 | - | - | - | 1 | -
$\dot{m}_{11}$ | 1 | (1, -1) | 1 | - | 1 | - | 1 | - | - | - | - | 1

Here a number indicates presence (its sign the direction of change), and - indicates absence.
The numbers in parentheses highlight that those parameters appear more than once in the ODE as a result of the emerging polynomial in the moments.

#### Step 2: Moment closure

In the context of this matrix, moment closure means dropping higher order ODEs (rows) and approximate higher order moments with lower order moments (columns).

&nbsp; | $\theta_1$ | $\theta_2$ | $m_{10}^{t_i}$ | $m_{01}^{t_i}$ | $m_{10}^{t_{i+1}}$ | $m_{01}^{t_{i+1}}$
-|-|-|-|-|-|-
$\dot{m}_{10}$ | 1 | -1 | 1 | - | 1 | -
$\dot{m}_{01}$ | - | 1 | 1 | 1 | - | 1

#### Step 3: Hidden variables effect

Let us diagnose the structural identifiability as a function of the hidden variables.

A system is identifiable if the number of constraining rows (ODE evaluated at time $t_i$) is at least equal to the number of unknowns (parameters and variables).
Below we only consider times $t_0$ and $t_1$, but we could carry on with $t_3$, $t_4$ and so on if necessary.
Similarly, we could derive ODEs of moments higher than $2$.

Overall, you will observe that tracking moments reduces the negative effect of hidden variables, which reduce identifiablity.
It follows that even higher order moments could increase it further, but the feasibility of such quality data is likely unrealistic.

**No hidden variables**

*Closed*:

&nbsp; | $\theta_1$ | $\theta_2$ | $m_{10}^{t_0}$ | $m_{01}^{t_0}$ | $m_{10}^{t_{1}}$ | $m_{01}^{t_{1}}$
-|-|-|-|-|-|-
$\dot{m}_{10}^{t_0}$ | 1 | -1 | - | - | - | -
$\dot{m}_{10}^{t_1}$ | 1 | -1 | - | - | - | -
$\dot{m}_{01}^{t_0}$ | - | 1 | - | - | - | -
$\dot{m}_{01}^{t_1}$ | - | 1 | - | - | - | -

Identifiable: 2 independent rows for 2 unknowns.

*Not closed*:

&nbsp; | $\theta_1$ | $\theta_2$ | $m_{10}^{t_0}$ | $m_{01}^{t_0}$ | $m_{20}^{t_0}$ | $m_{02}^{t_0}$ | $m_{11}^{t_0}$ | $m_{10}^{t_{1}}$ | $m_{01}^{t_{1}}$ | $m_{20}^{t_{1}}$ | $m_{02}^{t_{1}}$ | $m_{11}^{t_{1}}$
-|-|-|-|-|-|-|-|-|-|-|-|-
$\dot{m}_{10}^{t_0}$ | 1 | -1 | - | - | - | - | - | - | - | - | - | -
$\dot{m}_{10}^{t_1}$ | 1 | -1 | - | - | - | - | - | - | - | - | - | -
$\dot{m}_{01}^{t_0}$ | - | 1 | - | - | - | - | - | - | - | - | - | -
$\dot{m}_{01}^{t_1}$ | - | 1 | - | - | - | - | - | - | - | - | - | -
$\dot{m}_{20}^{t_0}$ | (2, 1) | (-2, 1) | - | - | - | - | - | - | - | - | - | -
$\dot{m}_{20}^{t_1}$ | (2, 1) | (-2, 1) | - | - | - | - | - | - | - | - | - | -
$\dot{m}_{02}^{t_0}$ | - | (2, 1) | -| - | - | - | - | - | - | - | - | -
$\dot{m}_{02}^{t_1}$ | - | (2, 1) | -| - | - | - | - | - | - | - | - | -
$\dot{m}_{11}^{t_0}$ | 1 | (1, -1) | - | - | - | - | - | - | - | - | - | -
$\dot{m}_{11}^{t_1}$ | 1 | (1, -1) | - | - | - | - | - | - | - | - | - | -

Identifiable: 5 independent rows for 2 unknowns (up to 3 additional unknowns could be inferred too).

**Hidden $n_1(t)$ - live cells**

*Closed*:

&nbsp; | $\theta_1$ | $\theta_2$ | $m_{10}^{t_0}$ | $m_{01}^{t_0}$ | $m_{10}^{t_1}$ | $m_{01}^{t_1}$ | $m_{10}^{t_2}$ | $m_{01}^{t_2}$
-|-|-|-|-|-|-|-|-
$\dot{m}_{10}^{t_0}$ | 1 | -1 | - | - | 1 | - | - | -
$\dot{m}_{10}^{t_1}$ | 1 | -1 | - | - | 1 | - | 1 | -
$\dot{m}_{01}^{t_0}$ | - | 1 | - | - | - | - | - | -
$\dot{m}_{01}^{t_1}$  | - | 1 | - | - | 1 | - | - | -

Identifiable: 4 independent rows for up to 4 unknowns.

*Not closed*:

&nbsp; | $\theta_1$ | $\theta_2$ | $m_{10}^{t_0}$ | $m_{01}^{t_0}$ | $m_{20}^{t_0}$ | $m_{02}^{t_0}$ | $m_{11}^{t_0}$ | $m_{10}^{t_1}$ | $m_{01}^{t_1}$ | $m_{20}^{t_1}$ | $m_{02}^{t_1}$ | $m_{11}^{t_1}$ | $m_{10}^{t_2}$ | $m_{01}^{t_2}$ | $m_{20}^{t_2}$ | $m_{02}^{t_2}$ | $m_{11}^{t_2}$
-|-|-|-|-|-|-|-|-|-|-|-|-|-|-|-|-|-
$\dot{m}_{10}^{t_0}$ | 1 | -1 | - | - | - | - | - | 1 | - | - | - | - | - | - | - | - | -
$\dot{m}_{10}^{t_1}$ | 1 | -1 | - | - | - | - | - | 1 | - | - | - | - | 1 | - | - | - | -
$\dot{m}_{01}^{t_0}$ | - | 1 | - | - | - | - | - | - | - | - | - | - | - | - | - | - | -
$\dot{m}_{01}^{t_1}$ | - | 1 | - | - | - | - | - | 1 | - | - | - | - | - | - | - | - | -
$\dot{m}_{20}^{t_0}$ | (2, 1) | (-2, 1) | - | - | - | - | - | - | - | 1 | - | - | - | - | - | - | -
$\dot{m}_{20}^{t_1}$ | (2, 1) | (-2, 1) | - | - | - | - | - | 1 | - | 1 | - | - | - | - | 1 | - | -
$\dot{m}_{02}^{t_0}$ | - | (2, 1) | - | - | - | - | - | - | - | - | - | - | - | - | - | - | -
$\dot{m}_{02}^{t_1}$ | - | (2, 1) | - | - | - | - | - | 1 | - | - | - | 1 | - | - | - | - | -
$\dot{m}_{11}^{t_0}$ | 1 | (1, -1) | - | - | - | - | - | - | - | - | - | 1 | - | - | - | - | -
$\dot{m}_{11}^{t_1}$ | 1 | (1, -1) | - | - | - | - | - | 1 | - | 1 | - | 1 | - | - | - | - | 1

Identifiable: 10 independent rows for up to 8 unknowns (up to 2 additional unknowns could be inferred too).

**Hidden $n_2(t)$ - dead cells**

*Closed*:

&nbsp; | $\theta_1$ | $\theta_2$ | $m_{10}^{t_0}$ | $m_{01}^{t_0}$ | $m_{10}^{t_1}$ | $m_{01}^{t_1}$ | $m_{10}^{t_2}$ | $m_{01}^{t_2}$
-|-|-|-|-|-|-|-|-
$\dot{m}_{10}^{t_0}$ | 1 | -1 | - | - | - | - | - | -
$\dot{m}_{10}^{t_1}$ | 1 | -1 | - | - | - | - | - | -
$\dot{m}_{01}^{t_0}$ | - | 1 | - | - | - | 1 | - | -
$\dot{m}_{01}^{t_1}$ | - | 1 | - | - | - | 1 | - | 1

Not identifiable: 3 independent rows for 4 unknowns.

*Not closed*:

&nbsp; | $\theta_1$ | $\theta_2$ | $m_{10}^{t_0}$ | $m_{01}^{t_0}$ | $m_{20}^{t_0}$ | $m_{02}^{t_0}$ | $m_{11}^{t_0}$ | $m_{10}^{t_1}$ | $m_{01}^{t_1}$ | $m_{20}^{t_1}$ | $m_{02}^{t_1}$ | $m_{11}^{t_1}$ | $m_{10}^{t_2}$ | $m_{01}^{t_2}$ | $m_{20}^{t_2}$ | $m_{02}^{t_2}$ | $m_{11}^{t_2}$
-|-|-|-|-|-|-|-|-|-|-|-|-|-|-|-|-|-
$\dot{m}_{10}^{t_0}$ | 1 | -1 | - | - | - | - | - | - | - | - | - | - | - | - | - | - | -
$\dot{m}_{10}^{t_1}$ | 1 | -1 | - | - | - | - | - | - | - | - | - | - | - | - | - | - | -
$\dot{m}_{01}^{t_0}$ | - | 1 | - | - | - | - | - | - | 1 | - | - | - | - | - | - | - | -
$\dot{m}_{01}^{t_1}$ | - | 1 | - | - | - | - | - | - | 1 | - | - | - | - | 1 | - | - | -
$\dot{m}_{20}^{t_0}$ | (2, 1) | (-2, 1) | - | - | - | - | - | - | - | - | - | - | - | - | - | - | -
$\dot{m}_{20}^{t_1}$ | (2, 1) | (-2, 1) | - | - | - | - | - | - | - | - | - | - | - | - | - | - | -
$\dot{m}_{02}^{t_0}$ | - | (2, 1) | - | - | - | - | - | - | - | - | 1 | - | - | - | - | - | -
$\dot{m}_{02}^{t_1}$ | - | (2, 1) | - | - | - | - | - | - | - | - | 1 | 1 | - | - | - | 1 | -
$\dot{m}_{11}^{t_0}$ | 1 | (1, -1) | - | - | - | - | - | - | - | - | - | 1 | - | - | - | - | -
$\dot{m}_{11}^{t_1}$ | 1 | (1, -1) | - | - | - | - | - | - | - | - | - | 1 | - | - | - | - | 1

Identifiable: 8 independent rows up to 8 unknowns (no additional unknowns could be inferred).

#### Step 4: Automatized diagnostic

Let us compare the previous results with the outcome of our `Julia` code and scale-up to more complicated models (e.g. consumer-resource models).

## 1. Import packages

In [124]:
using Catalyst
using MomentClosure
using Latexify

using LinearAlgebra

In [125]:
# # Install packages (just necessary for Google Colab)
# import Pkg
# Pkg.add("Catalyst")
# Pkg.add("MomentClosure")
# Pkg.add("Latexify")

## 2. Set model

In [126]:
# Uncomment the desired model and comment out the others, or define a new model using the style of the examples
# Avoid using μ as parameter, it is used by MomentClosure.jl to indicate moments.

# # Birth-death (only live cells)
# rn = @reaction_network begin
#     @parameters θ₁ θ₂
#     @species n₁(t)
#         θ₁, n₁ → 2n₁
#         θ₂, n₁ → 0
# end

# # Birth-death (only live cells - 2nd order polynomial)
# rn = @reaction_network begin
#     @parameters θ₁ θ₂ θ₃ θ₄
#     @species n₁(t)
#         θ₁ + θ₃*n₁, n₁ → 2n₁
#         θ₂ + θ₄*n₁, n₁ → 0
# end

# # Birth-death (live and dead cells)
# rn = @reaction_network begin
#     @parameters θ₁ θ₂
#     @species n₁(t) n₂(t)
#         θ₁, n₁ → 2n₁
#         θ₂, n₁ → n₂
# end

# # Birth-death (live and dead cells - 2nd order polynomial)
# rn = @reaction_network begin
#     @parameters θ₁ θ₂ θ₃ θ₄
#     @species n₁(t) n₂(t)
#         θ₁ + θ₃*n₁, n₁ → 2n₁
#         θ₂ + θ₄*n₁, n₁ → n₂
# end

# # 1 consumer, 1 resource model
# rn = @reaction_network begin
#     @parameters ρ₁ η₁ c₁
#     @species R₁(t) N₁(t)
#         ρ₁, 0 → R₁
#         η₁, N₁ → 0
#         c₁, N₁ + R₁ → 2N₁
# end

# # 2 consumers, 2 resources model
# rn = @reaction_network begin
#     @species R₁(t) R₂(t) N₁(t) N₂(t)
#         (ρ₁,ρ₂), 0 → (R₁,R₂)
#         (η₁,η₂), (N₁,N₂) → 0
#         (c₁₁,c₁₂), (N₁+R₁,N₁+R₂) → 2N₁
#         (c₂₁,c₂₂), (N₂+R₁,N₂+R₂) → 2N₂
# end

# # 6 consumers, 6 resources model
# rn = @reaction_network begin
#     @species R₀(t) R₁(t) R₂(t) R₃(t) R₄(t) R₅(t) N₀(t) N₁(t) N₂(t) N₃(t) N₄(t) N₅(t)
#         (ρ₀,ρ₁,ρ₂,ρ₃,ρ₄,ρ₅), 0 → (R₀,R₁,R₂,R₃,R₄,R₅)
#         (η₀,η₁,η₂,η₃,η₄,η₅), (N₀,N₁,N₂,N₃,N₄,N₅) → 0
#         (c₀₀,c₀₁,c₀₂,c₀₃,c₀₄,c₀₅), (N₀+R₀,N₀+R₁,N₀+R₂,N₀+R₃,N₀+R₄,N₀+R₅) → 2N₀
#         (c₁₀,c₁₁,c₁₂,c₁₃,c₁₄,c₁₅), (N₁+R₀,N₁+R₁,N₁+R₂,N₁+R₃,N₁+R₄,N₁+R₅) → 2N₁
#         (c₂₀,c₂₁,c₂₂,c₂₃,c₂₄,c₂₅), (N₂+R₀,N₂+R₁,N₂+R₂,N₂+R₃,N₂+R₄,N₂+R₅) → 2N₂
#         (c₃₀,c₃₁,c₃₂,c₃₃,c₃₄,c₃₅), (N₃+R₀,N₃+R₁,N₃+R₂,N₃+R₃,N₃+R₄,N₃+R₅) → 2N₃
#         (c₄₀,c₄₁,c₄₂,c₄₃,c₄₄,c₄₅), (N₄+R₀,N₄+R₁,N₄+R₂,N₄+R₃,N₄+R₄,N₄+R₅) → 2N₄
#         (c₅₀,c₅₁,c₅₂,c₅₃,c₅₄,c₅₅), (N₅+R₀,N₅+R₁,N₅+R₂,N₅+R₃,N₅+R₄,N₅+R₅) → 2N₅
# end;

# # 8 consumers, 8 resources model
# rn = @reaction_network begin
#     @species R₀(t) R₁(t) R₂(t) R₃(t) R₄(t) R₅(t) R₆(t) R₇(t) N₀(t) N₁(t) N₂(t) N₃(t) N₄(t) N₅(t) N₆(t) N₇(t)
#     (ρ₀,ρ₁,ρ₂,ρ₃,ρ₄,ρ₅,ρ₆,ρ₇), 0 → (R₀,R₁,R₂,R₃,R₄,R₅,R₆,R₇)
#     (η₀,η₁,η₂,η₃,η₄,η₅,η₆,η₇), (N₀,N₁,N₂,N₃,N₄,N₅,N₆,N₇) → 0
#     (c₀₀,c₀₁,c₀₂,c₀₃,c₀₄,c₀₅,c₀₆,c₀₇), (N₀+R₀,N₀+R₁,N₀+R₂,N₀+R₃,N₀+R₄,N₀+R₅,N₀+R₆,N₀+R₇) → 2N₀
#     (c₁₀,c₁₁,c₁₂,c₁₃,c₁₄,c₁₅,c₁₆,c₁₇), (N₁+R₀,N₁+R₁,N₁+R₂,N₁+R₃,N₁+R₄,N₁+R₅,N₁+R₆,N₁+R₇) → 2N₁
#     (c₂₀,c₂₁,c₂₂,c₂₃,c₂₄,c₂₅,c₂₆,c₂₇), (N₂+R₀,N₂+R₁,N₂+R₂,N₂+R₃,N₂+R₄,N₂+R₅,N₂+R₆,N₂+R₇) → 2N₂
#     (c₃₀,c₃₁,c₃₂,c₃₃,c₃₄,c₃₅,c₃₆,c₃₇), (N₃+R₀,N₃+R₁,N₃+R₂,N₃+R₃,N₃+R₄,N₃+R₅,N₃+R₆,N₃+R₇) → 2N₃
#     (c₄₀,c₄₁,c₄₂,c₄₃,c₄₄,c₄₅,c₄₆,c₄₇), (N₄+R₀,N₄+R₁,N₄+R₂,N₄+R₃,N₄+R₄,N₄+R₅,N₄+R₆,N₄+R₇) → 2N₄
#     (c₅₀,c₅₁,c₅₂,c₅₃,c₅₄,c₅₅,c₅₆,c₅₇), (N₅+R₀,N₅+R₁,N₅+R₂,N₅+R₃,N₅+R₄,N₅+R₅,N₅+R₆,N₅+R₇) → 2N₅
#     (c₆₀,c₆₁,c₆₂,c₆₃,c₆₄,c₆₅,c₆₆,c₆₇), (N₆+R₀,N₆+R₁,N₆+R₂,N₆+R₃,N₆+R₄,N₆+R₅,N₆+R₆,N₆+R₇) → 2N₆
#     (c₇₀,c₇₁,c₇₂,c₇₃,c₇₄,c₇₅,c₇₆,c₇₇), (N₇+R₀,N₇+R₁,N₇+R₂,N₇+R₃,N₇+R₄,N₇+R₅,N₇+R₆,N₇+R₇) → 2N₇
# end;

# # 10 consumers, 10 resources model
# rn = @reaction_network begin
#     @species R₀(t) R₁(t) R₂(t) R₃(t) R₄(t) R₅(t) R₆(t) R₇(t) R₈(t) R₉(t) N₀(t) N₁(t) N₂(t) N₃(t) N₄(t) N₅(t) N₆(t) N₇(t) N₈(t) N₉(t)
#     (ρ₀,ρ₁,ρ₂,ρ₃,ρ₄,ρ₅,ρ₆,ρ₇,ρ₈,ρ₉), 0 → (R₀,R₁,R₂,R₃,R₄,R₅,R₆,R₇,R₈,R₉)
#     (η₀,η₁,η₂,η₃,η₄,η₅,η₆,η₇,η₈,η₉), (N₀,N₁,N₂,N₃,N₄,N₅,N₆,N₇,N₈,N₉) → 0
#     (c₀₀,c₀₁,c₀₂,c₀₃,c₀₄,c₀₅,c₀₆,c₀₇,c₀₈,c₀₉), (N₀+R₀,N₀+R₁,N₀+R₂,N₀+R₃,N₀+R₄,N₀+R₅,N₀+R₆,N₀+R₇,N₀+R₈,N₀+R₉) → 2N₀
#     (c₁₀,c₁₁,c₁₂,c₁₃,c₁₄,c₁₅,c₁₆,c₁₇,c₁₈,c₁₉), (N₁+R₀,N₁+R₁,N₁+R₂,N₁+R₃,N₁+R₄,N₁+R₅,N₁+R₆,N₁+R₇,N₁+R₈,N₁+R₉) → 2N₁
#     (c₂₀,c₂₁,c₂₂,c₂₃,c₂₄,c₂₅,c₂₆,c₂₇,c₂₈,c₂₉), (N₂+R₀,N₂+R₁,N₂+R₂,N₂+R₃,N₂+R₄,N₂+R₅,N₂+R₆,N₂+R₇,N₂+R₈,N₂+R₉) → 2N₂
#     (c₃₀,c₃₁,c₃₂,c₃₃,c₃₄,c₃₅,c₃₆,c₃₇,c₃₈,c₃₉), (N₃+R₀,N₃+R₁,N₃+R₂,N₃+R₃,N₃+R₄,N₃+R₅,N₃+R₆,N₃+R₇,N₃+R₈,N₃+R₉) → 2N₃
#     (c₄₀,c₄₁,c₄₂,c₄₃,c₄₄,c₄₅,c₄₆,c₄₇,c₄₈,c₄₉), (N₄+R₀,N₄+R₁,N₄+R₂,N₄+R₃,N₄+R₄,N₄+R₅,N₄+R₆,N₄+R₇,N₄+R₈,N₄+R₉) → 2N₄
#     (c₅₀,c₅₁,c₅₂,c₅₃,c₅₄,c₅₅,c₅₆,c₅₇,c₅₈,c₅₉), (N₅+R₀,N₅+R₁,N₅+R₂,N₅+R₃,N₅+R₄,N₅+R₅,N₅+R₆,N₅+R₇,N₅+R₈,N₅+R₉) → 2N₅
#     (c₆₀,c₆₁,c₆₂,c₆₃,c₆₄,c₆₅,c₆₆,c₆₇,c₆₈,c₆₉), (N₆+R₀,N₆+R₁,N₆+R₂,N₆+R₃,N₆+R₄,N₆+R₅,N₆+R₆,N₆+R₇,N₆+R₈,N₆+R₉) → 2N₆
#     (c₇₀,c₇₁,c₇₂,c₇₃,c₇₄,c₇₅,c₇₆,c₇₇,c₇₈,c₇₉), (N₇+R₀,N₇+R₁,N₇+R₂,N₇+R₃,N₇+R₄,N₇+R₅,N₇+R₆,N₇+R₇,N₇+R₈,N₇+R₉) → 2N₇
#     (c₈₀,c₈₁,c₈₂,c₈₃,c₈₄,c₈₅,c₈₆,c₈₇,c₈₈,c₈₉), (N₈+R₀,N₈+R₁,N₈+R₂,N₈+R₃,N₈+R₄,N₈+R₅,N₈+R₆,N₈+R₇,N₈+R₈,N₈+R₉) → 2N₈
#     (c₉₀,c₉₁,c₉₂,c₉₃,c₉₄,c₉₅,c₉₆,c₉₇,c₉₈,c₉₉), (N₉+R₀,N₉+R₁,N₉+R₂,N₉+R₃,N₉+R₄,N₉+R₅,N₉+R₆,N₉+R₇,N₉+R₈,N₉+R₉) → 2N₉
# end;

# # Lotka-Volterra 2 species
# rn = @reaction_network begin
#     @species N₀(t) N₁(t)
#         (γ₀,γ₁), (N₀,N₁) → (2N₀,2N₁)
#         (α₀₀,α₁₁), (N₀,N₁) → 0
#         (α₀₁), (N₀+N₁) → 2N₀
#         (α₁₀), (N₁+N₀) → 2N₁
# end;

# # Lotka-Volterra 5 species
# rn = @reaction_network begin
#     @species N₀(t) N₁(t) N₂(t) N₃(t) N₄(t)
#         (γ₀,γ₁,γ₂,γ₃,γ₄), (N₀,N₁,N₂,N₃,N₄) → (2N₀,2N₁,2N₂,2N₃,2N₄)
#         (α₀₀,α₁₁,α₂₂,α₃₃,α₄₄), (N₀,N₁,N₂,N₃,N₄) → 0
#         (α₀₁,α₀₂,α₀₃,α₀₄), (N₀+N₁,N₀+N₂,N₀+N₃,N₀+N₄) → 2N₀
#         (α₁₀,α₁₂,α₁₃,α₁₄), (N₁+N₀,N₁+N₂,N₁+N₃,N₁+N₄) → 2N₁
#         (α₂₀,α₂₁,α₂₃,α₂₄), (N₂+N₀,N₂+N₁,N₂+N₃,N₂+N₄) → 2N₂
#         (α₃₀,α₃₁,α₃₂,α₃₄), (N₃+N₀,N₃+N₁,N₃+N₂,N₃+N₄) → 2N₃
#         (α₄₀,α₄₁,α₄₂,α₄₃), (N₄+N₀,N₄+N₁,N₄+N₂,N₄+N₃) → 2N₄
# end;

# # Michaelis-Menten
# rn = @reaction_network begin
#     @species E(t) S(t) ES(t) P(t)
#         (k₊, k₋), E + S <--> ES
#         (kₚ), ES → E + P
# end

# # Allosteric inhibition
# rn = @reaction_network begin
#     @species E(t) S(t) ES(t) P(t) I(t) EI(t)
#         (kₛ₊, kₛ₋), E + S <--> ES
#         (kᵢ₊, kᵢ₋), E + I <--> EI
#         (kₚ), ES → E + P
# end

# # Susceptible-Infected-Recovered model (constant population size)
# rn = @reaction_network begin
#     @species S(t) I(t) R(t)
#         β, S + I → 2I
#         γ, I → R
# end

# Susceptible-Infected-Recovered model (with birth-death)
rn = @reaction_network begin
    @species S(t) I(t) R(t)
        β, S + I → 2I
        γ, I → R
        θ₁, S → 2S
        θ₂, (S, I, R) → 0
end

# # Susceptible-Infected-Recovered-Deceased model (with birth-death)
# rn = @reaction_network begin
#     @species S(t) I(t) R(t) D(t)
#         β, S + I → 2I
#         γ, I → R
#         θ₁, S → 2S
#         (θ₂, θ₂+ϵ, θ₂), (S, I, R) → D
# end

# Susceptible-Exposed-Infected-Recovered-Deceased model (with birth-death)
# rn = @reaction_network begin
#     @species S(t) E(t) I(t) R(t) D(t)
#         β, S + I → E + I
#         α, E → I
#         γ, I → R
#         θ₁, S → 2S
#         (θ₂, θ₂, θ₂+ϵ, θ₂), (S, E, I, R) → D
# end

# # Microbial cross-feeding (CoSMO)
# rn = @reaction_network begin
#     @species A(t) L(t) a(t) l(t)
#     # @parameters bₐ bₗ dₐ dₗ cₗ cₐ rₐ rₗ
#         (bₐ, bₗ), (A+cₗl, L+cₐa) → (2A, 2L)
#         (dₐ, dₗ), (A, L) → (0, 0)
#         (rₐ, rₗ), (A, L) → (A+a, L+l)
# end

Model ##ReactionSystem#339:
Unknowns (3): see unknowns(##ReactionSystem#339)
  S(t)
  I(t)
  R(t)
Parameters (4): see parameters(##ReactionSystem#339)
  β
  γ
  θ₁
  θ₂

## 3. Set Structural Identifiability scenario

In [127]:
# Print variables and their index in the model
speciesmap(rn)

Dict{SymbolicUtils.BasicSymbolic{Real}, Int64} with 3 entries:
  R(t) => 3
  S(t) => 1
  I(t) => 2

In [128]:
# Establish settings for the Structuctural Identifiability analysis
hidden_variables = [2,3]    # List hidden variables. 
hidden_init_cond = []      # List hidden initial conditions

order_of_moments = 1            # Maximum order of moments.
moments_type = "raw"            # Options are: raw, central
closure = "normal"              # Moment closure. Options are: zero, normal, poisson, log-normal, gamma, and others.

depth = 1                   # Depth of time points
# all_init_cond_known = false      # All initial conditions known?
# n_tpoints = 2                   # Keep it unchanged. 1 is the minimum.

# Print warning if all variables are hidden.
if length(hidden_variables) >= length(species(rn))
    print("ERROR! Not all variables can be hidden.")
end

## 4. Get set of ODEs

In [129]:
# Derive moments ODEs
if moments_type == "raw"
    mom_eqs = generate_raw_moment_eqs(rn, order_of_moments, combinatoric_ratelaws=false)
elseif moments_type == "central"
    mom_eqs = generate_central_moment_eqs(rn, order_of_moments, combinatoric_ratelaws=false)
end
# latexify(mom_eqs) #|> print

# Use moment closure (zero, normal, poisson, log-normal, gamma, ...)
closed_mom_eqs = moment_closure(mom_eqs, closure)
latexify(closed_mom_eqs) #|> print

L"\begin{align*}
\frac{d\mu_{1 0 0}}{dt} &= \mu_{1 0 0} \theta_1 - \mu_{1 0 0} \theta_2 - \mu_{0 1 0} \mu_{1 0 0} \beta \\
\frac{d\mu_{0 1 0}}{dt} &=  - \mu_{0 1 0} \gamma - \mu_{0 1 0} \theta_2 + \mu_{0 1 0} \mu_{1 0 0} \beta \\
\frac{d\mu_{0 0 1}}{dt} &= \mu_{0 1 0} \gamma - \mu_{0 0 1} \theta_2
\end{align*}
"

In [130]:
# Extract parameters, moments and equations from the closed system of ODEs
params = parameters(closed_mom_eqs.odes)
moments = unknowns(closed_mom_eqs.odes)
moments_odes = equations(closed_mom_eqs.odes)

# Count number of parameters and ODEs
n_params = length(params)
n_moments = length(moments)
n_equations = length(moments_odes)

# Print summary
println("Number of parameters: \t", n_params)
println("Number of moments: \t", n_moments)
println("Number of ODEs: \t", n_equations)

Number of parameters: 	4
Number of moments: 	3
Number of ODEs: 	3


## 5. Proposed SI analysis 

### 5.1 Find number of time points per equation

In [131]:
# Rank of symbolic matrix using LU decomposition
# Based on sym_lu() from Symbolics.jl, linear_algebra.jl:11

function sym_lu_adhoc(A; check=true)
    SINGULAR = typemax(Int)
    m, n = size(A)
    F = map(x->x isa Symbolics.RCNum ? x : Symbolics.Num(x), A)
    minmn = min(m, n)
    p = Vector{LinearAlgebra.BlasInt}(undef, minmn)
    info = minmn#info = 0
    for k = 1:minmn
        kp = k
        amin = SINGULAR
        for i in k:m
            absi = Symbolics._iszero(F[i, k]) ? SINGULAR : Symbolics.nterms(F[i,k])
            if absi < amin
                kp = i
                amin = absi
            end
        end

        p[k] = kp

        if amin == SINGULAR && !(amin isa Symbolics.Symbolic) && (amin isa Number) #&& Symbolics.iszero(info)
            info -= 1#info = k
        end

        # swap
        for j in 1:n
            F[k, j], F[kp, j] = F[kp, j], F[k, j]
        end

        for i in k+1:m
            # F[i, k] = F[i, k] / F[k, k]
            # prevent divisions by zero from returning NaN
            if !Symbolics._iszero(F[k, k])
                F[i, k] = F[i, k] / F[k, k]
            else
                F[i, k] = 0
            end
        end
        for j = k+1:n
            for i in k+1:m
                F[i, j] = F[i, j] - F[i, k] * F[k, j]
            end
        end

    end
    return info
    # check && LinearAlgebra.checknonsingular(info)
    # Symbolics.LU(F, p, convert(LinearAlgebra.BlasInt, info))
end

sym_lu_adhoc (generic function with 1 method)

In [132]:
# ## Numeric test

# A₁ = [1 2; 2 4]                                     # rank 1 (singular)
# A₂ = [1 2 3; 2 4 6; 3 6 9]                          # rank 1 (singular)
# A₃ = [1 0; 0 1]                                     # rank 2
# A₄ = [1 0; 2 0]                                     # rank 1 (singular)
# A₅ = [1 2 3 4; 2 4 6 8; 1 6 9 12; 2 12 18 24]       # rank 2 (singular)

# for A in [A₁, A₂, A₃, A₄, A₅]
#     rank = sym_lu_adhoc(A)
#     println("Rank of A: $rank")
# end
# println()

# ## Symbolic test

# α = Symbolics.variables(:α, 1:3)

# B₁ = [α[1] 2*α[2]; 2*α[1] 4*α[2]]                                       # rank 1 (singular)
# B₂ = [α[1] 2*α[2] 3*α[3]; 2*α[1] 4*α[2] 6*α[3]; 3*α[1] 6*α[2] 9*α[3]]   # rank 1 (singular)
# B₃ = [α[1] 0; 0 α[2]]                                                   # rank 2
# B₄ = [α[1] 0; 2*α[1] 0]                                                 # rank 1 (singular)

# for B in [B₁, B₂, B₃, B₄]
#     rank = sym_lu_adhoc(B)
#     println("Rank of B: $rank")
# end

In [133]:
# Determine number of time points per equation (based on the rank) 
@variables t δ
τ = Symbolics.variables(:τ, 1:n_params+1)

println("Eq.\t", "# Par.\t", "Rank")
n_tpoints_per_eq = zeros(Int, n_equations)
eqs_all = []
for i in 1:n_equations

    params_in_eq = [param for param in params if Symbolics.occursin(param, moments_odes[i].rhs)]
    eqs = []
    for j in 1:length(params_in_eq)
        eq = substitute(moments_odes[i].rhs, Dict([t => τ[j]]))
        eq -= δ * substitute(moments[i], Dict([t => τ[j+1]])) - δ * substitute(moments[i], Dict([t => τ[j]]))
        append!(eqs, [eq])
        append!(eqs_all, [eq])
    end
    a, b, islinear = Symbolics.linear_expansion(eqs, params_in_eq)
    rank = sym_lu_adhoc(a; check=true)
    n_tpoints_per_eq[i] = rank
    println(i, "\t", length(params_in_eq), "\t", rank, " ")
end

n_tpoints = maximum(n_tpoints_per_eq)
if n_tpoints%2 == 0
    n_tpoints = Int8(depth*n_tpoints)
else
    n_tpoints = Int8(depth*(n_tpoints+1))
end
println("Time points per equation: ", n_tpoints_per_eq)

a, b, islinear = Symbolics.linear_expansion(eqs_all, params)
rank_all = sym_lu_adhoc(a; check=true)
println("\n# of parameters:\t", n_params)
println("Rank ODE system:\t", rank_all)
if rank_all >= n_params
    println("Identifiable without hidden variables")
else
    println("Nonidentifiable even without hidden variables")
end

Eq.	# Par.	Rank
1	3	3 
2	3	2 
3	2	2 
Time points per equation: [3, 2, 2]

# of parameters:	4
Rank ODE system:	4
Identifiable without hidden variables


### 5.2 Build Identifiability Matrix

In [134]:
# Create vector of labels with parameters and variables
labels = params
@variables t
for i in 0:n_tpoints
    # Include time tag from time zero (t0) to time n (e.g. t2) 
    time_tag = [substitute(moments[j], Dict([t => Meta.parse("t"*string(i))])) for j in 1:n_moments]
    labels = vcat(labels, time_tag)
end
labels;

In [135]:
# Check if a hidden variable appears in a symbolic moment (=0 is absence, >0 presence)
subindx_dict = Dict('₀'=>0, '₁'=>1, '₂'=>2, '₃'=>3, '₄'=>4, '₅'=>5, '₆'=>6, '₇'=>7, '₈'=>8, '₉'=>9)
function hidden_involvement(moment, hidden_variables)
    subindex = [subindx_dict[x] for x in string(moment)[3 .* hidden_variables]]
    return sum(subindex) != 0
end

hidden_involvement (generic function with 1 method)

In [148]:
# Build matrix of knowns (i.e. 0s) and unknowns (i.e. 1s) parameters and moments

# Empty matrix (we will fill in the unknowns with 1s)
si_matrix = zeros(Int8, n_tpoints * n_moments, n_params + (n_tpoints + 1) * n_moments)
# Loop over time points to fill in the matrix
for t in (0:n_tpoints-1) .* n_moments
    # Loop over the equations
    for i in 1:n_moments
        # Loop over the parameters
        for j in 1:n_params
            # Add 1 if the parameter appears on the RHS of the ODE
            si_matrix[t + i, j] = occursin(params[j], moments_odes[i].rhs)
        end
        # Loop over the moments
        for j in 1:n_moments
            # Add 1 if the moment (at time point t) appears on the RHS of the ODE
            si_matrix[t + i, n_params + t + j] = occursin(moments[j], moments_odes[i].rhs)
        end
        # Add 1 for the time points of a moment's derivative
        si_matrix[t + i, n_params + t + i] = 1
        si_matrix[t + i, n_params + t + i + n_moments] = 1
    end
end

# Without hidden variables only flag parameters as unknowns
if isempty(hidden_variables)
    known_columns = n_params .+ collect(1:(n_tpoints+1)*n_moments)
# With hidden variables flag them and parameters as unknowns
else
    # Initial conditions known?
    # all_init_cond_known = true
    # known_columns = ifelse(all_init_cond_known, n_params .+ collect(1:n_moments), [n_params+i for i in 1:n_moments if hidden_involvement(moments[i], hidden_variables) == 0])
    known_columns = isempty(hidden_init_cond) ? n_params .+ collect(1:n_moments) : [n_params+i for i in 1:n_moments if !hidden_involvement(moments[i], hidden_init_cond)]
    # Loop over all moments
    for i in 1:n_moments
        # Evaluate if hidden variable(s) are involved in the moment
        if !hidden_involvement(moments[i], hidden_variables)
            # Loop over time points
            for t in 1:n_tpoints
                # Save the flag
                push!(known_columns, n_params + t * n_moments + i)
            end
        end
    end
end
# Remove known variables (0s are knowns, 1s are unknowns)
si_matrix[:, known_columns] .= 0

# Identify rows of SI matrix to be considered based on independence
rows_to_keep = [] 
for i in 1:n_moments
    # With hidden variables in an equation all time evaluations are linearly independent
    if any(occursin.(moments[hidden_variables], moments_odes[i].rhs - moments_odes[i].lhs))
        t_p = 0:n_tpoints-1
    # Without hidden variables in an equation some time evaluations might be linearly dependent
    else
        t_p = 0:n_tpoints_per_eq[i]-1
    end
    append!(rows_to_keep, i .+ t_p .* n_moments)
end
rows_to_keep = sort(rows_to_keep)

# Print labels and matrix of knowns(0s)/unknowns(1s) 
println("[$(join(labels, " "))] ")
for i in eachrow(si_matrix)
    println("[$(join(i, " "))] ")
end
# Unknowns and constraints before the elimination steps
n_unknowns = sum(sum(si_matrix, dims=1) .!= 0)
n_constraints = length(rows_to_keep)
println("\nUnknowns: \t", n_unknowns)
println("Constraints: \t", n_constraints)
println("Rows to keep: \t", rows_to_keep)

[β γ θ₁ θ₂ μ₁₀₀(t0) μ₀₁₀(t0) μ₀₀₁(t0) μ₁₀₀(t1) μ₀₁₀(t1) μ₀₀₁(t1) μ₁₀₀(t2) μ₀₁₀(t2) μ₀₀₁(t2) μ₁₀₀(t3) μ₀₁₀(t3) μ₀₀₁(t3) μ₁₀₀(t4) μ₀₁₀(t4) μ₀₀₁(t4)] 
[1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] 
[1 1 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0] 
[0 1 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0] 
[1 0 1 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0] 
[1 1 0 1 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0] 
[0 1 0 1 0 0 0 0 1 1 0 0 1 0 0 0 0 0 0] 
[1 0 1 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0] 
[1 1 0 1 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0] 
[0 1 0 1 0 0 0 0 0 0 0 1 1 0 0 1 0 0 0] 
[1 0 1 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0] 
[1 1 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0] 
[0 1 0 1 0 0 0 0 0 0 0 0 0 0 1 1 0 0 1] 

Unknowns: 	12
Constraints: 	12
Rows to keep: 	Any[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


In [137]:
cols_to_keep = findall(!iszero, vec(sum(si_matrix[rows_to_keep, :], dims=1)))
# println(rows_to_keep)
i = length(cols_to_keep)
while !iszero(i)
    if sum(si_matrix[rows_to_keep, cols_to_keep[i]]) == 1
        row = [j for j in rows_to_keep if !iszero(si_matrix[j, cols_to_keep[i]])]
        unknowns_idx = [i for i in 1:(n_params+n_moments) if !iszero(si_matrix[row, i])]
        if !any(sum(si_matrix[rows_to_keep, unknowns_idx], dims=1) .== 1)
            rows_to_keep = setdiff(rows_to_keep, row)
        end
        # println(i, row, rows_to_keep)
    end
    i -= 1
end
# println(rows_to_keep)

# Print labels and matrix of knowns(0s)/unknowns(1s) 
println("[$(join(labels, " "))] ")
for i in eachrow(si_matrix[rows_to_keep,:])
    println("[$(join(i, " "))] ")
end
# Unknowns and constraints before the elimination steps
n_unknowns = sum(sum(si_matrix[rows_to_keep,:], dims=1) .!= 0)
n_constraints = length(rows_to_keep)
println("\nUnknowns: \t", n_unknowns)
println("Constraints: \t", n_constraints)
println("Rows to keep: \t", rows_to_keep)


[β γ θ₁ θ₂ μ₁₀₀(t0) μ₀₁₀(t0) μ₀₀₁(t0) μ₁₀₀(t1) μ₀₁₀(t1) μ₀₀₁(t1) μ₁₀₀(t2) μ₀₁₀(t2) μ₀₀₁(t2) μ₁₀₀(t3) μ₀₁₀(t3) μ₀₀₁(t3) μ₁₀₀(t4) μ₀₁₀(t4) μ₀₀₁(t4)] 
[1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] 
[1 1 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0] 
[1 0 1 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0] 
[1 1 0 1 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0] 
[1 0 1 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0] 
[1 1 0 1 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0] 
[1 0 1 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0] 

Unknowns: 	7
Constraints: 	7
Rows to keep: 	Any[1, 2, 4, 5, 7, 8, 10]


In [138]:
# # using Graphs
# # using Combinatorics

# n_columns = size(si_matrix,2)

# links = []
# for i in 1:n_columns
#     nz = findall(!iszero, si_matrix[rows_to_keep, i])#si_matrix[[1,3], i]
#     append!(links, collect(combinations(nz, 2)))
# end
# elist = Tuple.(sort(unique(links)));
# println(elist)
# g = SimpleGraph(Graphs.SimpleEdge.(elist));
# eulerian(g)
# # println(cycle_basis(g, 1))

# # elist = [[1,2],[2,3]];#,[2,4],[3,4],[4,1],[1,5]
# # elist = Tuple.(elist)
# # g = SimpleGraph(Graphs.SimpleEdge.(elist));
# # # cycle_basis(g)
# # eulerian(g,1)
# # # collect(all_simple_paths(g,1,2;cutoff=2))

In [139]:
# n_rows = size(si_matrix, 1)
# rows = rows_to_keep
# shared_rows = []
# while !isempty(rows)
#     i = rows[1]
#     i_rows = rows[findall(k -> sum(si_matrix[i, :] .* si_matrix[k, :]) != 0, rows)]
#     push!(shared_rows, i_rows)
#     rows = setdiff(rows, i_rows)
# end
# display(shared_rows)

n_columns = size(si_matrix, 2)
shared_rows = [rows_to_keep[findall(!iszero, si_matrix[rows_to_keep, i])] for i in 1:n_columns]#[rows_to_keep[findall(si_matrix[rows_to_keep, i] .== 1)] for i in 1:n_columns]
shared_rows = filter(!isempty, shared_rows)#filter(x -> !isempty(x), shared_rows)
merges = 1
while merges > 0
    merges = 0
    i = 1
    while i < length(shared_rows)
        j = i+1
        while j <= length(shared_rows)
            if !isempty(intersect(shared_rows[i], shared_rows[j]))
                shared_rows[i] = sort(union(shared_rows[i], shared_rows[j]))
                deleteat!(shared_rows, j)
                merges += 1
            else
                j += 1
            end
        end
        i += 1
    end
end
println(shared_rows)

for rows_set in shared_rows
    nz_ind = findall(!iszero, vec(sum(si_matrix[rows_set, :], dims=1)))#findall(x -> x != 0, vec(sum(si_matrix[rows_set, :], dims=1)))
    n_constraints = length(rows_set)
    n_unknowns = length(nz_ind)

    nz_ind_par = [i for i in nz_ind if i <= (n_params+n_moments)]
    println("\n[$(join(labels[nz_ind_par], " "))]")
    if n_unknowns <= n_constraints
        println("Identifiable.")
    else
        println("Nonidentifiable.")
    end
    println("Unknowns:\t", n_unknowns)
    println("Constraints:\t", n_constraints)
end

Vector{Any}[[1, 2, 4, 5, 7, 8, 10]]

[β γ θ₁ θ₂]
Identifiable.
Unknowns:	7
Constraints:	7


In [140]:
# using Combinatorics

# function solvable_system(rrd_comb)
#     test_m = sum(si_matrix_solved[rrd_comb, :], dims=1)
#     return sum(!iszero, test_m) == req_dim
# end

# n_columns = size(si_matrix, 2)
# si_matrix_solved = copy(si_matrix)
# rows_not_eliminated = rows_to_keep

# identifiable_, identifiable_others = [], []
# nonidentifiable_ = findall(!iszero, vec(sum(si_matrix, dims=1)))

# req_dim = 1
# while req_dim <= 26
#     rows_req_dim = [j for j in rows_not_eliminated if sum(si_matrix_solved[j, :]) <= req_dim]
#     # println(req_dim, " ", identifiable_, rows_not_eliminated)

#     rrd_combs = collect(combinations(rows_req_dim, req_dim))
#     # println(req_dim, " ", rrd_combs)

#     test_rrd_comb = rrd_combs[solvable_system.(rrd_combs)]

#     if isempty(test_rrd_comb)
#         req_dim += 1
#     else
#         println(test_rrd_comb[1])
#         test_m = sum(si_matrix_solved[test_rrd_comb[1], :], dims=1)
#         unknowns_rrd_comb = findall(!iszero, vec(test_m))
#         si_matrix_solved[:, unknowns_rrd_comb] .= 0

#         rows_not_eliminated = [i for i in rows_not_eliminated if sum(si_matrix_solved[i,:]) != 0]

#         ident_ = [labels[i] for i in unknowns_rrd_comb if i <= (n_params+n_moments)]
#         push!(identifiable_, ident_)
#         ident_others = [labels[i] for i in unknowns_rrd_comb if i > (n_params+n_moments)]
#         push!(identifiable_others, ident_others)
#         nonidentifiable_ = setdiff(nonidentifiable_, unknowns_rrd_comb)
        
#         req_dim = 1
#     end
#     if isempty(intersect(nonidentifiable_, 1:(n_params+n_moments)))
#         break
#     end
# end
# nonidentifiable_ = [labels[i] for i in nonidentifiable_ if i <= (n_params+n_moments)];

# # @time solvable_system.(rrd_combs)

# println("\nIdentifiable:")
# for set_i in identifiable_
#     println("[$(join(set_i, " "))] ")
# end

# println("\nIdentifiable (others):")
# for set_i in identifiable_others
#     println("[$(join(set_i, " "))] ")
# end

# println("\nNonidentifiable:")
# println("[$(join(nonidentifiable_, " "))] ")

### # Old methodology

In [141]:
# # Dictionary breaking the ODEs in coefficients, parameters and term bases (products of moments in the equations)

# # Create empty dictionary and list of terms in the equations
# params_dict = Dict([i => Dict([param => Dict() for param in params]) for i in 1:n_moments])
# term_bases = []

# # Loop over the equations
# for i in 1:n_moments
#     # Extract RHS of equation
#     eq = moments_odes[i].rhs
#     # Break equation into terms if it has more than one
#     terms = (operation(eq) == +) ? arguments(eq) : [eq]
#     # Loop over the terms in the equation
#     for term in terms
#         # Loop over the parameters
#         for param in params
#             # Check if the parameter appears in the term
#             if occursin(param, term)
#                 # Keep only coefficient and moment of the term
#                 term_factors = factors(substitute(term,  Dict([param => 1])))
#                 # Extract coefficient of the term (no having a coefficient implies it is 1)
#                 coefficient = term_factors[Symbolics.degree.(term_factors) .== 0]
#                 coefficient = isempty(coefficient) ? 1 : coefficient[1]
#                 # Extract moment(s) of the term (multiplying the moments of the term)
#                 term_base = prod(term_factors[Symbolics.degree.(term_factors) .> 0])
#                 params_dict[i][param][term_base] = Symbolics.symbolic_to_float(coefficient)
#                 # If a new product of moments ("terms base") was observed, store it and remove redundancies
#                 append!(term_bases, term_base)
#                 term_bases = unique(term_bases)
#             end
#         end
#     end
# end

# # Print dictionary
# params_dict # term_bases

In [142]:
# # Build matrix of knowns (i.e. 0s) and unknowns (i.e. 1s) parameters and moments

# # Empty matrix (we will fill in the unknowns with 1s)
# si_matrix = zeros(n_tpoints * n_moments, n_params + (n_tpoints + 1) * n_moments)
# # Loop over the time_points
# for t in 0:n_tpoints-1
#     # List time points (t0, t1, t2, ...) of a moment
#     t_offset = t * n_moments
#     # Loop over equations
#     for i in 1:n_moments
#         # Loop over the parameters
#         for j in 1:n_params
#             # Add 1 if the parameter appears on the RHS of the ODE
#             si_matrix[i + t_offset, j] = occursin(params[j], moments_odes[i].rhs)
#         end
#         # Add 1 for the time points of a moment's derivative
#         si_matrix[i + t_offset, i + n_params + t_offset] = 1
#         si_matrix[i + t_offset, i + n_params + t_offset + n_moments] = 1
#         # Loop over the moments
#         for j in 1:n_moments
#             # Add 1 if the moment (at time point t) appears on the RHS of the ODE
#             si_matrix[i + t_offset, j + n_params + t_offset] = occursin(moments[j], moments_odes[i].rhs)
#         end
#     end
# end

# # Without hidden variables only flag parameters as unknowns
# if isempty(hidden_variables)
#     known_columns = collect(n_params+1:n_params+(n_tpoints+1)*n_moments)
# # Flag hidden variables as unknowns
# else
#     # Initial conditions known?
#     known_columns = ifelse(all_init_cond_known, collect(n_params+1:n_params+n_moments), [n_params+i for i in 1:n_moments if hidden_involvement(moments[i], hidden_variables) == 0])
#     # Loop over all moments
#     for i in 1:n_moments
#         # Evaluate if hidden variable(s) are involved in the moment
#         if !hidden_involvement(moments[i], hidden_variables)
#             # Loop over time points
#             for t in 1:n_tpoints
#                 # Save the flag
#                 push!(known_columns, n_params + t * n_moments + i)
#             end
#         end
#     end
# end
# # Remove not hidden variables (0s are knowns, 1s are unknowns)
# si_matrix[:, known_columns] .= 0

# # Print labels and knowns(0s)/unknowns(1s) matrix
# println("[$(join(labels, " "))] ")
# for i in eachrow(si_matrix)
#     println("[$(join(i, " "))] ")
# end
# # Unknowns and constraints before the elimination steps
# n_unknowns = sum(sum(si_matrix, dims=1) .!= 0)
# n_constraints = sum(sum(si_matrix, dims=2) .!= 0)
# println("Unknowns: \t", n_unknowns)
# println("Constraints: \t", n_constraints)

In [143]:
# # Remove rows of known/unknow matrix based on their independence

# # Identify unique rows (i.e. non-repeated rows)
# uniq_rows = unique(si_matrix, dims=1)
# # Find and store index of repeated rows
# flagged_sets = []
# for uniq_row in eachrow(uniq_rows)
#     conflicts = findall(row -> row == uniq_row, eachrow(si_matrix))
#     length(conflicts) > 1 ? append!(flagged_sets, [conflicts]) : nothing
# end

# # Start with all rows kept
# rows_to_keep = 1:n_moments*n_tpoints

# # Loop over the groups of repeated rows to check if they should be removed
# for flagged_set in flagged_sets
#     # TODO: Compute det(equation). Keep as many rows (t0, t1, t2, etc) of the eq. as params if det != 0. 
#     # Sort by params, use substitute (param => 1) to only keep coefficients, break into terms and compute det from those terms.

#     # If flagged, only keep row of same moment ODE at t0
#     idx_to_remove = flagged_set[flagged_set .> n_moments]
#     rows_to_keep = setdiff(rows_to_keep, idx_to_remove)
#     flagged_set = setdiff(flagged_set, idx_to_remove)

#     # If flagged, evaluate if two rows have the same model structure
#     params_combs = []
#     # Loop over flagged rows sets
#     for moment in flagged_set[flagged_set .<= n_moments]# for moment in flagged_set
#         params_comb = zeros(n_params, length(term_bases))
#         # Loop over parameters
#         for i in 1:n_params
#             # Loop over term bases
#             for j in 1:length(term_bases)
#                 # Use the ODE dictionary to fill a matrix of parameters vs. term bases
#                 params_comb[i, j] = get(params_dict[moment][params[i]], term_bases[j], 0)
#             end
#         end
#         # Append the matrix of parameters vs. term bases
#         append!(params_combs, [params_comb])
#     end

#     # Remove rows whose matrix of parameters vs. term bases appears more than once
#     idx_to_keep = flagged_set[[findfirst(x -> x == params_comb, params_combs) for params_comb in unique(params_combs)]]
#     # ### TODO: Explain
#     # for i in idx_to_keep
#     #     for j in i .+ n_moments*(1:n_tpoints-1)
#     #         print(i,j, j in flagged_set, det_eqs[i], j in flagged_set && det_eqs[i])
#     #         if j in flagged_set && det_eqs[i]
#     #             append!(idx_to_keep, j)
#     #         end
#     #     end
#     # end
#     # ###
#     idx_to_remove = setdiff(flagged_set, idx_to_keep)
#     rows_to_keep = setdiff(rows_to_keep, idx_to_remove)
    
# end

# # Print rows of knowns/unknowns matrix that will be kept
# rows_to_keep

In [144]:
# # Elimination algorithm using the matrix of knowns/unknowns to solve for the parameters

# # Lists to store parameters/variables solved and equations used to do it
# solution_path_param, solution_path_eq = [], []
# # Matrix being solved
# si_matrix_solved = copy(si_matrix)
# # Rows still not eliminated
# rows_not_eliminated = rows_to_keep
# # Remaining parameters to solve for
# params_to_solve = collect(1:n_params)

# # Function to eliminate rows by 
# function elimination(si_matrix_solved, elimination_sets, params_to_solve, solution_path_param, solution_path_eq, rows_not_eliminated)
#     # Add index of rows to list of used rows
#     append!(solution_path_eq, [elimination_sets])
#     # Loop over sets of repeated rows
#     for elimination_set in elimination_sets
#         # Ignore rows that are all zero
#         if sum(si_matrix_solved[elimination_set[1], :]) != 0
#             # Identify non-zero columns (i.e. containing unknown parameter/variables)
#             non_zero_columns = findall(x -> x != 0, si_matrix_solved[elimination_set[1], :])
#             # Make non-zero columns all zero (i.e. the unknown parameter/variables are now solved)
#             si_matrix_solved[:, non_zero_columns] .= 0
#             # Remove paramaters that have been solved
#             params_to_solve = setdiff(params_to_solve, non_zero_columns)
#             # Save index of parameters/variables solved
#             append!(solution_path_param, [non_zero_columns])
#         end
#         # Remove rows that have been used to solve
#         rows_not_eliminated = setdiff(rows_not_eliminated, elimination_set)
#     end
#     return si_matrix_solved, params_to_solve, solution_path_param, solution_path_eq, rows_not_eliminated
# end

# # Loop until there is no parameter to solve for
# while !isempty(params_to_solve)
#     # Empty list of sets of rows to eliminate
#     elimination_sets = []
#     # Find unique rows in the known/unknown matrix
#     uniq_rows = unique(si_matrix_solved[rows_not_eliminated,:], dims=1)
    
#     # Loop over the unique rows
#     for uniq_row in eachrow(uniq_rows)
#         # Identify rows sharing parameters/variables
#         sharing_rows = [i for i in rows_not_eliminated if si_matrix_solved[i,:] == uniq_row]
#         # Count number of rows sharing parameters/variables and their unknowns
#         n_constraints = length(sharing_rows)
#         n_unknowns = sum(sum(si_matrix_solved[sharing_rows, :], dims=1) .!= 0)
#         # If there are more constraints than unknowns (i.e. overdetermined) ...
#         if n_constraints > n_unknowns
#             # Keep only the first constraints that equal the number of unknowns
#             sharing_rows = sharing_rows[1:n_unknowns]
#             n_constraints = length(sharing_rows)
#         end
#         # If there are sharing rows add them to the list of rows sets to be eliminated
#         length(sharing_rows) > 1 && (n_constraints == n_unknowns) ? append!(elimination_sets, [sharing_rows]) : nothing
#     end
    
#     # Include rows of knowns/unknowns matrix with a single unknown to solve for
#     append!(elimination_sets, [i for i in rows_not_eliminated if sum(si_matrix_solved[i,:]) == 1])

#     # If there are no rows groups flagged for elimination ...
#     if isempty(elimination_sets)
#         n_constraints = length(rows_not_eliminated)
#         n_unknowns = sum(sum(si_matrix_solved[rows_not_eliminated, :], dims=1) .!= 0)
#         # More constraints that unknowns means remaining parameters/variables can be solved
#         if n_constraints >= n_unknowns # TODO: Check that all remaining rows are "connected".
#             println( "Identifiable (2): \t", string.(params[params_to_solve]))
#         # Fewer constraints that unknowns means remaining parameters/variables can not be solved
#         else
#             println( "Nonidentifiable: \t", string.(params[params_to_solve]))
#         end
#         # Stop the elimination algorithm
#         break
#     # If there are rows flagged for elimination ...
#     else
#         # Delete the rows and change their unknowns to knowns
#         si_matrix_solved, params_to_solve, solution_path_param, solution_path_eq, rows_not_eliminated = elimination(si_matrix_solved, elimination_sets, params_to_solve, solution_path_param, solution_path_eq, rows_not_eliminated)
#     end
# end
# # Print identifiable parameters/variables
# isempty(solution_path_param) ? nothing : println("Identifiable (1): \t", [labels[i] for i in solution_path_param])
# # Print sequence of equations eliminated to solve for parameters/variables
# isempty(solution_path_eq) ? nothing : println("Solution path (eqs): \t", solution_path_eq)

# # Print labels and knowns(0s)/unknowns(1s) matrix at the last elimination step
# println("\n[$(join(labels, " "))] ")
# for i in eachrow(si_matrix_solved)
#     println("[$(join(i, " "))] ")
# end
# # Unknowns and constraints remaining at the last elimination step
# n_unknowns = sum(sum(si_matrix_solved[rows_not_eliminated, :], dims=1) .!= 0)
# n_constraints = sum(sum(si_matrix_solved[rows_not_eliminated, :], dims=2) .!= 0)
# println("\nUnknowns: \t", n_unknowns)
# println("Constraints: \t", n_constraints)

Note that this approach does not distinguish between global and local identifiability (currently), but goes from transition rates to identifiability results quickly.

## 6. Benchmark: StructuralIdentifiability.jl

### 6.1 Import packages

In [145]:
# Import packages
using StructuralIdentifiability
using ModelingToolkit
using Logging

### 6.2 Set model

In [146]:
# Define symbolic variables
@independent_variables t
@variables y[1:n_moments]

# Trasform ODESystem from MomentClosure.jl to a ModelingToolkit.jl System
model_sijl = System(moments_odes, t, name = :model_sijl)

# Specify vector of observed moments and initial conditions
observed = [y[i] ~ moments[i] for i in 1:n_moments]
init_cond = isempty(hidden_init_cond) ? moments : [moments[i] for i in 1:n_moments if !hidden_involvement(moments[i], hidden_init_cond)]
if !isempty(hidden_variables)
    observed = [y[i] ~ moments[i] for i in 1:n_moments if !hidden_involvement(moments[i], hidden_variables)]
    # if !all_init_cond_known
    #     init_cond = [moments[i] for i in 1:n_moments if !hidden_involvement(moments[i], hidden_variables)]
    # end
end

println(model_sijl)
println("observed: ", observed)
println("init. cond.: ", init_cond)

Model model_sijl:
Equations (3):
  3 standard: see equations(model_sijl)
Unknowns (3): see unknowns(model_sijl)
  μ₁₀₀(t)
  μ₀₁₀(t)
  μ₀₀₁(t)
Parameters (4): see parameters(model_sijl)
  θ₂
  β
  θ₁
  γ
observed: Equation[y[1] ~ μ₁₀₀(t)]
init. cond.: SymbolicUtils.BasicSymbolic{Real}[μ₁₀₀(t), μ₀₁₀(t), μ₀₀₁(t)]


### 6.3 Diagnose

In [147]:
# StructuralIdentifiability.jl assesment
SIjl_output = assess_identifiability(model_sijl, measured_quantities = observed, known_ic = init_cond, prob_threshold=0.99, loglevel=Logging.Error)

OrderedCollections.OrderedDict{SymbolicUtils.BasicSymbolic{Real}, Symbol} with 7 entries:
  μ₁₀₀(0) => :globally
  μ₀₁₀(0) => :globally
  μ₀₀₁(0) => :globally
  θ₂      => :nonidentifiable
  β       => :globally
  θ₁      => :nonidentifiable
  γ       => :nonidentifiable